# Support Vector Machines — Classification Foundations

**Goal:** build an interview-ready, DataCamp-style SVM exercise that makes scaling, margins, kernels and hyperparameter tuning visible.

Dataset: scikit-learn Breast Cancer Wisconsin diagnostic dataset. This is a learning/foundation notebook, not a clinical system.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

RANDOM_STATE = 42
data = load_breast_cancer(as_frame=True)
X, y = data.data.copy(), data.target.copy()
print(X.shape)
display(X.head())
display(y.value_counts(normalize=True).rename('share'))


## 1. Data split and baseline

SVMs are sensitive to feature scale, so scaling stays **inside** each pipeline to prevent train/test leakage.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print('Dummy accuracy:', accuracy_score(y_test, baseline_pred))
print('Dummy F1:', f1_score(y_test, baseline_pred))


## 2. Linear SVM — maximum-margin classification

A linear SVM learns a separating hyperplane with a wide margin. The absolute decision score gives a useful view of distance from the boundary.


In [ ]:
linear_svm = Pipeline([
    ('scale', StandardScaler()),
    ('model', LinearSVC(C=1.0, dual='auto', random_state=RANDOM_STATE, max_iter=20000)),
])
linear_svm.fit(X_train, y_train)
linear_pred = linear_svm.predict(X_test)
linear_scores = linear_svm.decision_function(X_test)
print('Linear SVM accuracy:', accuracy_score(y_test, linear_pred))
print('Linear SVM F1:', f1_score(y_test, linear_pred))
display(pd.Series(np.abs(linear_scores), name='absolute_margin').describe())


## 3. RBF kernel + cross-validation

The RBF kernel allows non-linear boundaries. `C` controls the penalty for mistakes; `gamma` controls how local each training example's influence is.


In [ ]:
rbf = Pipeline([('scale', StandardScaler()), ('model', SVC(kernel='rbf'))])
search = GridSearchCV(
    rbf,
    {'model__C': [0.1, 1, 10, 100], 'model__gamma': ['scale', 0.001, 0.01, 0.1]},
    cv=5, scoring='f1', n_jobs=-1,
)
search.fit(X_train, y_train)
rbf_pred = search.predict(X_test)
print(search.best_params_)
print('Tuned RBF accuracy:', accuracy_score(y_test, rbf_pred))
print('Tuned RBF F1:', f1_score(y_test, rbf_pred))
print('Support vectors per class:', search.best_estimator_.named_steps['model'].n_support_)


In [ ]:
comparison = pd.DataFrame([
    {'model':'Dummy', 'accuracy':accuracy_score(y_test, baseline_pred), 'f1':f1_score(y_test, baseline_pred)},
    {'model':'Linear SVM', 'accuracy':accuracy_score(y_test, linear_pred), 'f1':f1_score(y_test, linear_pred)},
    {'model':'Tuned RBF SVM', 'accuracy':accuracy_score(y_test, rbf_pred), 'f1':f1_score(y_test, rbf_pred)},
]).sort_values('f1', ascending=False)
display(comparison.round(4))

ConfusionMatrixDisplay.from_predictions(y_test, rbf_pred, display_labels=data.target_names)
plt.title('Tuned RBF SVM — confusion matrix')
plt.show()


## 4. Two-dimensional PCA inspection view

PCA is used **only for visualisation** here; the fitted SVMs above use the complete feature set.


In [ ]:
X_scaled = StandardScaler().fit_transform(X)
projection = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
plt.figure(figsize=(8, 5))
for label, name in enumerate(data.target_names):
    mask = y.to_numpy() == label
    plt.scatter(projection[mask, 0], projection[mask, 1], alpha=0.6, label=name)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Dataset projected to 2D')
plt.legend()
plt.show()


## Takeaway

This foundation covers **scaling, maximum margins, linear vs RBF kernels, `C`, `gamma`, support vectors, cross-validation, confusion-matrix analysis and inference**.

The professional NLP project applies `LinearSVC` to high-dimensional TF-IDF text and adds probability calibration and confidence routing.
